<div align="center">
      <h1>Introduction to k-Nearest Neighbors</h1>
      <hr>
</div>

## Before You Start

In this notebook, you will build a k-NN classifier from scratch using **NumPy only** for the implementation. Work with NumPy arrays and basic array operations; **do not use PyTorch**, SciPy distance helpers, or a ready-made k-NN classifier such as scikit-learn's `KNeighborsClassifier`. Follow the constraints in each assignment, including avoiding `np.linalg.norm()` for the distance computations.

The supplied Matplotlib plots and scikit-learn accuracy helper are fine to use; the NumPy-only requirement applies to your k-NN implementation.

Keep the [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html) and [broadcasting guide](https://numpy.org/doc/stable/user/basics.broadcasting.html) handy. Reading the documentation and trying small examples will help you understand the operations you use.


In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['image.interpolation'] = 'nearest'

%load_ext autoreload
%autoreload 2

## Understanding Our Dataset

### What We're Working With
For this introduction to k-NN, we'll use a carefully designed 2D classification dataset. This dataset contains:
- **Three distinct classes** of points in a 2-dimensional feature space
- **Points generated from Gaussian distributions** - each class has its own distribution center and spread
- **A linearly separable structure** - meaning the classes can be separated by straight lines

### Why This Dataset?
Starting with a 2D dataset allows us to:
1. **Visualize everything** - we can plot the data points and see the decision boundaries
2. **Build intuition** - understand how k-NN makes decisions before moving to complex, high-dimensional data
3. **Debug easily** - if something goes wrong, we can see it immediately in the plots

Let's load and explore our data:

In [ ]:
import os
import numpy as np

from utils import Data2DVisualizer, dataset_stats

notebook_dir = os.getcwd()
dataset_path = os.path.join(notebook_dir, 'data', 'datasets', 'linearly_separable.npz')

# Load the dataset
dataset = np.load(dataset_path)

X_train, y_train = dataset['X_train'], dataset['y_train']
X_val, y_val = dataset['X_val'], dataset['y_val']
X_test, y_test = dataset['X_test'], dataset['y_test']

vis = Data2DVisualizer((X_train, y_train), (X_val, y_val), (X_test, y_test))
vis.show_dataset()

num_features, num_classes, num_samples = dataset_stats(X_train, y_train, X_val, y_val, X_test, y_test, verbose=True)

## Understanding k-Nearest Neighbors

### The Core Intuition
Imagine you move to a new neighborhood and want to know if it's safe. What would you do? You'd probably look at the houses around you - if most nearby houses seem secure and well-maintained, you'd feel the neighborhood is safe. **This is exactly how k-NN works!**

### How k-NN Makes Predictions
The k-Nearest Neighbors algorithm classifies data points based on a simple principle: **"You are who your neighbors are."**

Here's the step-by-step process:
1. **Choose k** - the number of neighbors to consider (e.g., k=3 means look at 3 nearest neighbors)
2. **Calculate distances** - measure how far the new point is from all training points
3. **Find k nearest neighbors** - select the k training points that are closest
4. **Vote** - the most common class among these k neighbors becomes the prediction

### Key Characteristics
- **Non-parametric**: k-NN doesn't assume any underlying data distribution
- **Lazy learning**: In our implementation, `train()` stores the training examples and labels; distance calculations and voting happen during prediction
- **Instance-based**: Stores all training data and uses it directly for predictions

### When to Use k-NN?
k-NN works best when:
- Local patterns matter more than global structure
- The dataset isn't too large (since we need to compute distances to all points)
- Features are on similar scales (or properly normalized)
- The feature space has relatively few relevant dimensions

### A Limitation: High-Dimensional Features
k-NN often performs poorly when feature vectors have many dimensions, especially when many features are noisy or irrelevant. With a fixed number of training examples, points become sparse in the larger space, and distances to nearby and faraway points can become more similar. The nearest neighbors may then be less informative about a point's class. This is part of the **curse of dimensionality**.

High dimensionality does not always make k-NN fail: the quality of the features and the underlying structure of the data matter too. Selecting useful features or reducing dimensionality can help.

For further reading, see the [nearest-neighbor overview](https://scikit-learn.org/stable/modules/neighbors.html). Course materials are available on the [Deep Learning Essentials course page](https://cw.fel.cvut.cz/wiki/courses/becm33dpl/start).

## Implementing k-Nearest Neighbors

Now let's build our own k-NN classifier from scratch! This will help you understand exactly what happens under the hood.

### Your Implementation Tasks
Navigate to `src/assignments/knn_classifier.py` where you'll find the `KNNClassifier` class skeleton. You need to complete two core methods:

1. **Assignment 1.1**: `_compute_distances` - Calculate Euclidean distances between test points and all training points
2. **Assignment 1.2**: `_predict_labels` - Use the distances to find k nearest neighbors and predict labels

### Testing Your Implementation
Once you've written your code, run the assignment tests:

```bash
$ hw1-test-assignments
```

At this stage, look for passing results for Assignments 1.1 and 1.2. This command runs all homework tests, so tests for tasks you have not implemented yet may still fail. The relevant part of the output should look like this:
```bash
================= TESTING ASSIGNMENTS =================

assignment_1_1:
        PASSED!

assignment_1_2:
        PASSED!
```

Let's see your implementation in action:

In [ ]:
from sklearn.metrics import accuracy_score
from assignments import KNNClassifier

# Create and train the classifier
knn = KNNClassifier(k=3, vectorized=False)
knn.train(X_train, y_train)

# Predict the labels of the given samples
y_pred = knn.predict(X_val)

# Compute the accuracy of the classifier, the accuracy should be around 0.98
print(f'Accuracy: {accuracy_score(y_val, y_pred):.3f}')

## The Power of Vectorization

### Why Vectorization Matters
The loop-based implementation computes a distance for every pair of query and training points. As the number of points grows, the nested Python loops can become a bottleneck. Vectorization lets NumPy perform these calculations with less Python overhead.

Vectorization still computes all pairwise distances: for `M` query points and `N` training points, the distance matrix has shape `(M, N)`. Keep its memory cost in mind when working with larger datasets.

### What is Vectorization?
Vectorization leverages:
1. **NumPy's optimized C implementations** - much faster than Python loops
2. **CPU SIMD instructions** - process multiple data points simultaneously
3. **Better memory access patterns** - improved cache utilization

### The Math Behind Vectorized Distance Computation
Instead of computing distances one by one, we can use matrix operations. The Euclidean distance can be rewritten using the identity:

$$||x - y||^2 = ||x||^2 + ||y||^2 - 2 \cdot x^T y$$

This allows us to:
1. Compute all squared norms at once
2. Use matrix multiplication for the dot products
3. Combine results efficiently

### Your Task
**Assignment 1.3**: Implement the vectorized version in the `_compute_distances_vectorized(X)` method within `src/assignments/knn_classifier.py`, using NumPy operations without explicit loops. Your goal is to compute the same distances, up to floating-point rounding, more efficiently. Then rerun `hw1-test-assignments` and check that Assignment 1.3 passes too.

In [ ]:
knn_vectorized = KNNClassifier(k=3, vectorized=True)
knn_vectorized.train(X_train, y_train)

# Predict the labels of the given samples
y_pred = knn_vectorized.predict(X_val)

# Compute the accuracy of the classifier. You can check that the accuracy is the same as for the non-vectorized implementation.
print(f'Accuracy: {accuracy_score(y_val, y_pred):.3f}')

## Performance Comparison: Loops vs. Vectors

### The Speedup Test
Let's quantify the performance difference between our two implementations. We'll use the same dataset and measure the time it takes to make predictions.

**What to expect:**
- The vectorized implementation is usually faster, but the speedup depends on the dataset size, hardware, and NumPy installation
- Timings can vary between runs, especially for small datasets; repeat the measurement before drawing conclusions
- The distance matrices should agree up to floating-point rounding, and the predicted labels should match on this dataset. Matching accuracy alone does not verify that the predictions are identical

In [ ]:
from time import time


def measure_time(classifier: KNNClassifier, X: np.ndarray) -> float:
    """ Measures the running time of predicting the labels of the given samples.
    
    Args:
        classifier: The classifier to measure the running time of.
        X: The dataset to measure the running time on.
    
    Returns:
        The running time of the classifier in seconds.
    """

    start = time()
    _ = classifier.predict(X)
    end = time()

    return end - start


print(f'Non-vectorized implementation took {measure_time(knn, X_val):.3f} seconds')
print(f'Vectorized implementation took {measure_time(knn_vectorized, X_val):.3f} seconds')

## Visualizing Decision Boundaries

### What Are Decision Boundaries?
Decision boundaries are the **borders between regions** in feature space where the classifier changes its prediction from one class to another. For k-NN:
- Each point in space is classified based on its k nearest neighbors
- The boundary forms where the "vote" changes from one class to another
- The shape depends on the value of k and the distribution of training points

### Impact of k on Boundaries
- **Small k (e.g., k=1)**: Creates complex, detailed boundaries that closely follow training data
- **Large k (e.g., k=50)**: Produces smoother, more general boundaries
- **Trade-off**: Small k → more flexible but prone to noise; Large k → more stable but might miss local patterns

Let's visualize how our k-NN classifier carves up the feature space:

In [ ]:
# Visualize the decision boundaries of the k-Nearest Neighbors classifier
vis.show_decision_boundaries(knn_vectorized, h=0.03)

## Evaluating Classifier Performance

### Why Test Set Performance Matters
We've trained our classifier and it works on the validation set, but the **true test** of any machine learning model is how well it performs on completely unseen data. This is why we kept a separate test set that we haven't touched until now.

### Understanding Accuracy
**Accuracy** is the simplest classification metric:
$$\text{Accuracy} = \frac{\text{Number of Correct Predictions}}{\text{Total Number of Predictions}}$$

While simple, accuracy gives us a quick sense of overall performance. However, remember:
- It treats all errors equally (misclassifying a cat as a dog vs. as a plane)
- It can be misleading with imbalanced datasets
- Other metrics (precision, recall, F1) might be more appropriate depending on your task

Let's see how our k-NN classifier performs on the held-out test set:

In [ ]:
y_pred = knn.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')

## Where k-NN Fails

k-NN has no real training step: `train()` only **stores** the labeled training points. All the work happens at prediction time, when a query point takes the majority label of its k closest stored points. k-NN therefore relies on two assumptions:

1. **Nearby points share a label.** The closest stored points are reliable evidence about the query.
2. **Distance measures similarity.** Points that are close in Euclidean distance are also similar in the ways that matter for the task.

Each experiment below breaks one of these assumptions. In every experiment, k-NN stores the training points, and we evaluate it on two kinds of queries:

- **Training points.** This is optimistic: each training point is stored, so it finds itself at distance 0 and votes for its own label. With k=1, training accuracy is therefore 100%, unless two identical points have different labels.
- **Validation points.** They are not stored, so their accuracy shows how well k-NN predicts points it has not seen.

The experiments use k=3. Try k=1 and compare training and validation accuracy.

### 1. Noisy Features

This experiment breaks the first assumption. We move every training and validation point by an independent random offset and keep its label:

$$x'_j = x_j + \epsilon_j, \qquad \epsilon_j \sim U(-a s_j, a s_j).$$

Here $a$ is the noise strength and $s_j$ is the standard deviation of feature $j$ in the original training points. At $a=0$, nothing changes. As $a$ grows, the classes overlap, so a point's nearest neighbors increasingly come from other classes.

Run the cell after completing Assignments 1.1–1.3. Each column is one noise level. The top row shows the stored training points, and the bottom row shows the validation points. The background is the predicted class, and black circles mark validation mistakes. All panels use the same axes.

Feel free to change noise levels and k the number of nearest neighbors.


In [ ]:
from utils.knn_experiments import run_noise_experiment

noise_results = run_noise_experiment(
    X_train, y_train, X_val, y_val,
    noise_levels=[0, 0.5, 1.5, 3],
    k=3,
    seed=0,
)


**Think about it:** Where do mistakes appear as noise increases? Why does training accuracy stay above validation accuracy? Hint: which point is always among a training point's neighbors?


### 2. Features in Different Units

This experiment breaks the second assumption. We multiply the x-coordinate of every training and validation point by $a$, as if we changed its units, for example from meters to millimeters. The labels stay the same, but the squared distance becomes

$$d_a^2(x,z) = a^2(x_1-z_1)^2 + (x_2-z_2)^2.$$

As $a$ grows, differences in x dominate the distance and y is effectively ignored: the nearest neighbors are simply the points with the most similar x. **This is why we normalize features before using k-NN: a feature's units should not decide how much it matters.** We normalize each feature as

$$z_j = \frac{x'_j-\mu_j}{s_j},$$

where $\mu_j$ and $s_j$ are the mean and standard deviation of feature $j$ in the training points. Validation points are normalized with the same values.

The **top row** shows the raw features with equal units on both axes, which is the geometry k-NN actually sees. As $a$ grows, the data flatten into a thin horizontal line and the decision boundaries become vertical: only x decides the class. The **bottom row** shows the same data after normalization, which looks the same for every $a$. Dots are training points, triangles are validation points, and circles mark validation mistakes.

In the accuracy plot, the color shows which points are evaluated: **black for training** and **orange for validation**. The line style shows the features: **solid for raw** and **dashed for normalized**.


In [ ]:
from utils.knn_experiments import run_scaling_experiment

scaling_results = run_scaling_experiment(
    X_train, y_train, X_val, y_val,
    scale_factors=[1, 10, 100, 10000],
    k=3,
)


**Think about it:** At x × 10,000, which two classes are confused, and why can't x alone separate them? Why does normalization give the same result for every $a$? What would happen if you multiplied both coordinates by the same positive number?


### 3. Too Many Features

k-NN needs stored points close to every query. This experiment shows that, as the number of features grows, the nearest point is no longer close, so the first assumption fails even without noise.

We generate a new dataset in which we choose the number of features $D$. Each coordinate is sampled uniformly from $[-1,1]$, and the label says whether the coordinates sum to a positive number:

$$y = \begin{cases}1 & \text{if } x_1 + \cdots + x_D > 0,\\0 & \text{otherwise.}\end{cases}$$

All features matter equally, there is no noise, and the true boundary is a flat hyperplane. This should be an easy problem.

Why do more features hurt? A box covering 10% of each coordinate's range occupies $0.1^D$ of the space. In 2D, that is 1% of the space, so about 10 of 1,000 training points fall inside. In 10D, it is $10^{-10}$ of the space. Even with 1M training points, the box is almost always empty. The number of training points needed to keep neighbors close grows exponentially with $D$.

We compare **1k, 10k, 100k, and 1M training points** for 2 to 500 features. The smaller training sets are subsets of the larger ones, and all sizes use the same 1,000 validation points. Everything is repeated with three random seeds.

The cell searches all 1M points in batches and can take a few minutes. Its figures are saved in the notebook, so you can read them without rerunning it.


In [ ]:
from utils.knn_experiments import run_dimension_experiment

dimension_results = run_dimension_experiment(
    training_sizes=[1_000, 10_000, 100_000, 1_000_000],
    dimensions=[2, 10, 50, 100, 500],
    validation_size=1_000,
    seeds=[100, 101, 102],
    k=3,
    batch_size=5_000,
)


**Reading the figures:**

1. **Accuracy** for each training-set size. Lines are means over three repeats, shading is one standard deviation, and the dashed line is random guessing.
2. **Why accuracy drops.** Both panels use the same axis and colors as the first figure:
   - *Left:* the distance to the nearest training point, as a percentage of the mean distance to all training points. Near 0%, the nearest point almost touches the query. Near 100%, the nearest point is almost as far away as the mean distance.
   - *Right:* the share of the 3 nearest neighbors that have the same label as the validation point. At 50%, the neighbors are no better than a coin flip, so their vote carries no information.

**Think about it:**

- How much does a larger training set help with 2 features? How much does it help with 500?
- With 500 features, is the nearest training point actually near?
- How do the nearest-distance and neighbor-label plots explain the accuracy drop?
- The true boundary is a hyperplane. Could a model that learns a hyperplane use the same data more effectively?
